In [2]:
import platform, sys
arch = platform.machine()
py_version = sys.version_info
print(f"Architecture: {arch}")
print(f"Python Version: {py_version.major}.{py_version.minor}.{py_version.micro}")

if arch != "arm64":
    raise EnvironmentError(
        "You need native arm64 Python"
    )

if py_version >= (3,14):
    raise EnvironmentError(
        "Pythin version Not supprted"
    )

print("Environment Ok!")

Architecture: arm64
Python Version: 3.12.12
Environment Ok!


In [3]:
%pip install -q --upgrade mlx-lm datasets huggingface_hub rouge_score

Note: you may need to restart the kernel to use updated packages.


In [1]:
import mlx_lm
print(f"mlx-lm version: {mlx_lm.__version__}")
from packaging.version import Version
if Version(mlx_lm.__version__) < Version("0.21.0"):
    print("mlx-lm may be too old for gemma 4")
else:
    print("mlx-lm version OK!")

mlx-lm version: 0.31.3
mlx-lm version OK!


In [4]:
from getpass import getpass
from huggingface_hub import login, whoami

HF_TOKEN = getpass("Paste your Hugging Face token (starts with hf_): ").strip()
if not HF_TOKEN:
    raise ValueError("No token provided. Generate one at https://huggingface.co/settings/tokens")
if not HF_TOKEN.startswith("hf_"):
    raise ValueError("That does not look like a valid Hugging Face token.")

login(token=HF_TOKEN, add_to_git_credential=False)
user = whoami()
print("✅ Logged in to Hugging Face Hub")

✅ Logged in to Hugging Face Hub


In [13]:
MODEL_ID = "mlx-community/gemma-4-e2b-it-4bit"
MAX_SEQ_LENGTH = 256    # covers 99.9% of your data, down from 1024
                        # smaller = faster training + less memory per batch
                        # the 1 outlier at 1127 tokens will just be truncated

ITERS = 600             # your dataset is 2367 examples, 90% = ~2130 train
                        # at effective batch 8: 2130/8 ≈ 266 steps per epoch
                        # 600 steps ≈ ~2.8 epochs — good starting point
LORA_RANK = 16
LORA_LAYERS = 24
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
DATA_FILE = "instructions_upsampled.txt"
DATA_DIR = "data"
ADAPTER_DIR = "adapters/survival_qlora"
FUSED_DIR = "survival_model_merged"
GGUF_DIR = "survival_gguf"
print("Config loaded")
print(f"Mode: {MODEL_ID}")
print(f"LoRA rank : {LORA_RANK} | LoRA layers: {LORA_LAYERS}")
print(f"Iters : {ITERS} | Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Lerning rate: {LEARNING_RATE}")

Config loaded
Mode: mlx-community/gemma-4-e2b-it-4bit
LoRA rank : 16 | LoRA layers: 24
Iters : 600 | Effective batch: 16
Lerning rate: 0.0002


In [6]:
import re
import os
import json
import random

def parse_qa(filepath):
    with open(filepath,"r",encoding="utf-8") as f:
        text = f.read()

    pattern = re.compile(
        r"Q:\s*(.+?)\nA:\s*(.+?)(?=\nQ:|\Z)",
        re.DOTALL
    )
    pairs = []
    for match in pattern.finditer(text):
        question = match.group(1).strip()
        answer = match.group(2).strip()
        if len(question) > 10 and len(answer) > 20:
            pairs.append({"question":question,"answer":answer})
    return pairs
pairs = parse_qa(DATA_FILE)
print(f"Parsed {len(pairs)} QnA pairs")
print(f"Sample entry:")
print(f"Q: {pairs[0]['question'][:100]}")
print(f"A: {pairs[0]['answer'][:150]}...")

Parsed 2631 QnA pairs
Sample entry:
Q: How long can food stay safe in the refrigerator during a power outage?
A: Your refrigerator will keep food cold for about 4 hours if you keep the door closed. A full freezer will maintain its temperature for approximately 48...


In [7]:
from transformers import AutoTokenizer

SYSTEM_PROMPT = (
    "You are SurvivalGuide, an expert survival assistant. "
    "You provide clear, practical, actionable advice on survival situations "
    "including power outages, water shortages, natural disasters, wilderness "
    "survival, first aid, and emergency preparedness. "
    "Your answers are concise, direct, and prioritise safety above all else. "
    "When someone is in immediate danger, lead with the most critical action first."
)

print("Loading tokenizer for chat template formatting...")
tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-4-e2b-it",
    trust_remote_code=True
)
print("Tokenizer Loaded!")

def format_example(pair):
    """
    Format a Q&A pair using Gemma 4's actual chat template.
    
    The result looks like:
        <start_of_turn>system
        You are SurvivalGuide...<end_of_turn>
        <start_of_turn>user
        How do I find water?<end_of_turn>
        <start_of_turn>model
        Look for...<end_of_turn>
    """
    messages = [
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":pair["question"]},
        {"role":"assistant","content":pair["answer"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
random.seed(42)
random.shuffle(pairs)
split_idx = int(len(pairs)*0.9)
train_pairs = pairs[:split_idx]
valid_pairs = pairs[split_idx:]

os.makedirs(DATA_DIR,exist_ok=True)
def write_jsonl(pairs_list, filepath):
    with open(filepath,"w",encoding="utf-8") as f:
        for pair in pairs_list:
            record = {"text":format_example(pair)}
            f.write(json.dumps(record,ensure_ascii=False)+"\n")
write_jsonl(train_pairs,f"{DATA_DIR}/train.jsonl")
write_jsonl(valid_pairs,f"{DATA_DIR}/valid.jsonl")

print("Dataset written: ")
print(f"Train: {len(train_pairs)} examples -> {DATA_DIR}/train.jsonl")
print(f"Valid: {len(valid_pairs)} examples -> {DATA_DIR}/valid.jsonl")
print(f"Test: 50 examples -> {DATA_DIR}/test.jsonl")

print("\nFormatted sample: ")
print("-"*60)
sample_text = format_example(train_pairs[0])
print(sample_text[:500] + ("..." if len(sample_text) > 500 else ""))
print("-"*60)

Loading tokenizer for chat template formatting...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Tokenizer Loaded!
Dataset written: 
Train: 2367 examples -> data/train.jsonl
Valid: 264 examples -> data/valid.jsonl
Test: 50 examples -> data/test.jsonl

Formatted sample: 
------------------------------------------------------------
<bos><|turn>system
You are SurvivalGuide, an expert survival assistant. You provide clear, practical, actionable advice on survival situations including power outages, water shortages, natural disasters, wilderness survival, first aid, and emergency preparedness. Your answers are concise, direct, and prioritise safety above all else. When someone is in immediate danger, lead with the most critical action first.<turn|>
<|turn>user
How do I create a saw from materials?<turn|>
<|turn>model
Use wire...
------------------------------------------------------------


In [14]:
import statistics
print("Measuring token lengths across training set...")
lengths = []
with open(f"{DATA_DIR}/train.jsonl") as f:
    for line in f:
        record = json.loads(line)
        n_tokens = len(tokenizer.encode(record["text"]))
        lengths.append(n_tokens)
lengths.sort()
p95 = lengths[int(0.95 * len(lengths))]
p99 = lengths[int(0.99 * len(lengths))]

print(f"\n Token length distribution (training set, n={len(lengths)}):")
print(f"Min: {min(lengths)}")
print(f"Median: {statistics.median(lengths):.0f}")
print(f"Mean: {statistics.mean(lengths):.0f}")
print(f"P95: {p95} <- ideal MAX_SEQ_LENGTH")
print(f"P99: {p99}")
print(f"Max: {max(lengths)}")

if p95 > MAX_SEQ_LENGTH:
    print(f"Your P95({p95}) exceed MAX_SEQ_LENGTH ({MAZ_SEQ_LENGTH}).")
    print(f"Consider increasing MAX_SEQ_LENGTH")
else:
    pct_covered = sum(1 for l in lengths if l <= MAX_SEQ_LENGTH)/len(lengths)*100
    print(f"\n MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} coveres {pct_covered:.1f}% of training examples")
    
est_mem_gb = (BATCH_SIZE * MAX_SEQ_LENGTH * 2 * 4) / (1024**3)
print(f"Rough memory per batch: ~{est_mem_gb:.2f} GB (activations only, not weights)")

# Add this to Cell 7, after the length analysis
outliers = []
with open(f"{DATA_DIR}/train.jsonl") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        n = len(tokenizer.encode(record["text"]))
        if n > 500:
            outliers.append((i, n, record["text"][:300]))

print(f"Found {len(outliers)} outlier(s) over 500 tokens:")
for idx, n_tokens, preview in outliers:
    print(f"\n  Line {idx}: {n_tokens} tokens")
    print(f"  Preview: {preview}...")

Measuring token lengths across training set...

 Token length distribution (training set, n=2367):
Min: 105
Median: 120
Mean: 144
P95: 197 <- ideal MAX_SEQ_LENGTH
P99: 211
Max: 1127

 MAX_SEQ_LENGTH=256 coveres 99.8% of training examples
Rough memory per batch: ~0.00 GB (activations only, not weights)
Found 3 outlier(s) over 500 tokens:

  Line 505: 1127 tokens
  Preview: <bos><|turn>system
You are SurvivalGuide, an expert survival assistant. You provide clear, practical, actionable advice on survival situations including power outages, water shortages, natural disasters, wilderness survival, first aid, and emergency preparedness. Your answers are concise, direct, an...

  Line 1541: 1127 tokens
  Preview: <bos><|turn>system
You are SurvivalGuide, an expert survival assistant. You provide clear, practical, actionable advice on survival situations including power outages, water shortages, natural disasters, wilderness survival, first aid, and emergency preparedness. Your answers are con

In [18]:
import yaml

os.makedirs(ADAPTER_DIR, exist_ok=True)

lora_config = {
    "model": MODEL_ID,
    "fine_tune_type": "lora",
    "num_layers": LORA_LAYERS,
    "rank": LORA_RANK,
    "scale": LORA_RANK * 2,
    "dropout": 0.05,

    "iters": ITERS,
    "batch_size": BATCH_SIZE,
    "grad_accumulation_steps": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,

    "lr_schedule": {
        "name": "cosine_decay",
        "warmup": int(ITERS * 0.05),
        "decay_steps": ITERS,
    },

    "data": DATA_DIR,
    "max_seq_length": MAX_SEQ_LENGTH,

    "mask_prompt_in_loss": True,
    "grad_checkpoint": True,

    "adapter_path": ADAPTER_DIR,
    "save_every":200,
    "steps_per_eval":100,
    "steps_per_report":25,
    "val_batches": 25,
    "seed": 42,
}

CONFIG_PATH = "qlora_config.yaml"
with open(CONFIG_PATH,"w") as f:
    yaml.dump(lora_config,f,default_flow_style=False,sort_keys=False)
print("Config written to {CONFIG_PATH}")
print("\nFull config: ")
print("-"*50)
with open(CONFIG_PATH) as f:
    print(f.read())

Config written to {CONFIG_PATH}

Full config: 
--------------------------------------------------
model: mlx-community/gemma-4-e2b-it-4bit
fine_tune_type: lora
num_layers: 24
rank: 16
scale: 32
dropout: 0.05
iters: 600
batch_size: 4
grad_accumulation_steps: 4
learning_rate: 0.0002
lr_schedule:
  name: cosine_decay
  warmup: 30
  decay_steps: 600
data: data
max_seq_length: 256
mask_prompt_in_loss: true
grad_checkpoint: true
adapter_path: adapters/survival_qlora
save_every: 200
steps_per_eval: 100
steps_per_report: 25
val_batches: 25
seed: 42



In [20]:
import subprocess, sys, time
cmd = [
    sys.executable, "-m", "mlx_lm", "lora",
    "--config", CONFIG_PATH,
    "--train",
]
print("Starting QLoRA fine-tuning...")
print(f"Command: {' '.join(cmd)}\n")
print("-"*60)
print("What to expect: ")
print("First ~50 steps: loss may be high (10-15) - normal")
print("Steps 50-200: loss should drop rapidly")
print("Steps 200-1000: gradual improvement, watch eval loss")
print("Total time: ~45-90 minutes")
print("-"*60 + "\n")

start_time = time.time()

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

log_lines = []
for line in process.stdout:
    print(line,end="",flush=True)
    log_lines.append(line)

process.wait()
elapsed = time.time()

with open("training.log","w") as f:
    f.writelines(log_lines)

print(f"\n{'-'*60}")
if process.returncode == 0:
    print(f"Training complete in {elapsed/60:.1f} minutes")
    print(f"Adapter saved to: {ADAPTER_DIR}/")
else:
    print(f"Training failed (exit code {process.returncode})")
    print("Check the error above. Common fixes: ")
    print("-OOM: Reduce BATCH_SIZE or LORA_LAYERS in Cell 4")
    print("-Architecture error: pip install --upgrade mlx-lm")

Starting QLoRA fine-tuning...
Command: /Users/shreeyansvichare/miniforge3/bin/python -m mlx_lm lora --config qlora_config.yaml --train

------------------------------------------------------------
What to expect: 
First ~50 steps: loss may be high (10-15) - normal
Steps 50-200: loss should drop rapidly
Steps 200-1000: gradual improvement, watch eval loss
Total time: ~45-90 minutes
------------------------------------------------------------

Loading configuration file qlora_config.yaml
Loading pretrained model

Fetching 8 files: 100%|██████████| 8/8 [16:42<00:00, 125.28s/it]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/shreeyansvichare/miniforge3/lib/python3.12/site-packages/mlx_lm/__main__.py", line 6, in <module>
    cli.main()
  File "/Users/shreeyansvichare/miniforge3/lib/python3.12/site-packages/mlx_lm/cli.py", line 40, in main
    submodule.main()
  File "/Users/shreeyans

In [21]:
# Add this as a temporary cell and run it
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "mlx_lm", "lora", "--config", "qlora_config.yaml", "--train"],
    capture_output=True,
    text=True
)
print("STDOUT:")
print(result.stdout[-3000:])  # last 3000 chars
print("\nSTDERR:")
print(result.stderr[-3000:])  # last 3000 chars

STDOUT:
Loading configuration file qlora_config.yaml
Loading pretrained model


STDERR:
ers.27.self_attn.k_proj.scales,
language_model.model.layers.27.self_attn.k_proj.weight,
language_model.model.layers.27.self_attn.v_proj.biases,
language_model.model.layers.27.self_attn.v_proj.scales,
language_model.model.layers.27.self_attn.v_proj.weight,
language_model.model.layers.28.self_attn.k_norm.weight,
language_model.model.layers.28.self_attn.k_proj.biases,
language_model.model.layers.28.self_attn.k_proj.scales,
language_model.model.layers.28.self_attn.k_proj.weight,
language_model.model.layers.28.self_attn.v_proj.biases,
language_model.model.layers.28.self_attn.v_proj.scales,
language_model.model.layers.28.self_attn.v_proj.weight,
language_model.model.layers.29.self_attn.k_norm.weight,
language_model.model.layers.29.self_attn.k_proj.biases,
language_model.model.layers.29.self_attn.k_proj.scales,
language_model.model.layers.29.self_attn.k_proj.weight,
language_model.model.layers.29.self_attn

In [1]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj="/Users/shreeyansvichare/llama.cpp/survival-q4_k_m.gguf",
    path_in_repo="/Users/shreeyansvichare/llama.cpp/survival-q4_k_m.gguf",
    repo_id="Shreyy2305/survival-gguf"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Shreyy2305/survival-gguf/commit/37a5763cddbded9630b2c90e56b5db5ce4cb3ea3', commit_message='Upload /Users/shreeyansvichare/llama.cpp/survival-q4_k_m.gguf with huggingface_hub', commit_description='', oid='37a5763cddbded9630b2c90e56b5db5ce4cb3ea3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Shreyy2305/survival-gguf', endpoint='https://huggingface.co', repo_type='model', repo_id='Shreyy2305/survival-gguf'), pr_revision=None, pr_num=None)